# Jordan-Wide 6-Model Equal-Weight Ensemble — SSP5-8.5
## Notebook 20

Generates six Jordan-wide NetCDF files using the equal-weight 6-model ensemble for SSP5-8.5. Logic identical to Notebook 08; only the model directory and output path differ.

$$P_{\text{ens6}}(i,j,t) = \frac{1}{6}\sum_{k=1}^{6} M_k(i,j,t)$$

**Output:** `ssp5-8.5 output\6 ensemble models\Jordan_6model_ensemble_ssp585_{period}.nc`

## 1. Import Libraries

In [1]:
import warnings
import datetime
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point
import time as _time

warnings.filterwarnings("ignore")

print("Libraries loaded.")
for lib, mod in [("numpy", np), ("pandas", pd), ("xarray", xr), ("geopandas", gpd)]:
    print(f"  {lib:<12}: {mod.__version__}")

Libraries loaded.
  numpy       : 2.1.3
  pandas      : 2.2.3
  xarray      : 2025.12.0
  geopandas   : 1.0.1


## 2. Configuration

In [2]:
SHAPEFILE = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\jordan.basins\surface.basins.for.jordan\Jo.shp"
)

# SSP5-8.5 model NC files (filename prefix: merged_)
MODEL_DIR = Path(r"D:\RICAAR8.5\merge\Precipitation")

OUTPUT_DIR = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\ssp5-8.5 output\6 ensemble models"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    "CMCC-CM2-SR5",
    "CNRM-ESM2-1",
    "EC-Earth3-Veg",
    "IPSL-CM6A-LR",
    "MPI-ESM1-2-LR",
    "NorESM2-MM",
]
N_MODELS = len(MODELS)

PR_VAR       = "prAdjust"
SYRIA_BASINS = {"YARMOUK (SYRIA)", "AZRAQ (SYRIA)", "AMMAN ZARQA (SYRIA)"}

PERIODS = [
    ("1961_2070", 1961, 2070),
    ("1995_2014", 1995, 2014),
    ("2015_2020", 2015, 2020),
    ("2021_2040", 2021, 2040),
    ("2041_2060", 2041, 2060),
    ("2061_2070", 2061, 2070),
]

print("Configuration loaded.")
print(f"  Models     : {N_MODELS} (equal weight — all models contribute to every cell)")
print(f"  Model dir  : {MODEL_DIR}")
print(f"  Shapefile  : {SHAPEFILE}")
print(f"  Output dir : {OUTPUT_DIR}")
print()
print("Six output periods:")
for label, y0, y1 in PERIODS:
    print(f"  {label}  ({y0}–{y1})")

Configuration loaded.
  Models     : 6 (equal weight — all models contribute to every cell)
  Model dir  : D:\RICAAR8.5\merge\Precipitation
  Shapefile  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\jordan.basins\surface.basins.for.jordan\Jo.shp
  Output dir : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ssp5-8.5 output\6 ensemble models

Six output periods:
  1961_2070  (1961–2070)
  1995_2014  (1995–2014)
  2015_2020  (2015–2020)
  2021_2040  (2021–2040)
  2041_2060  (2041–2060)
  2061_2070  (2061–2070)


## 3. Load Shapefile — Jordan Domain Mask

In [3]:
gdf_all    = gpd.read_file(SHAPEFILE)
gdf_jordan = gdf_all[~gdf_all["BASIN_NAME"].isin(SYRIA_BASINS)].copy().reset_index(drop=True)

print(f"Total shapefile features : {len(gdf_all)}")
print(f"Jordan-only retained     : {len(gdf_jordan)}")
print()
print("Jordan basins included in domain mask:")
for _, row in gdf_jordan.iterrows():
    print(f"  {row['BASIN_NAME']}")

Total shapefile features : 19
Jordan-only retained     : 16

Jordan basins included in domain mask:
  HAMMAD
  YARMOUK (JORDAN)
  J.VALLEY-YARMOUK TRIANGLE
  JORDAN VALLY (JORDAN)
  N.R.S.W
  AZRAQ (JORDAN)
  AMMAN ZARQA (JORDAN)
  S.R.S.W
  MUJIB
  D.S.R.S.W
  W. ARABA NORTH
  HASA
  JAFER
  WADI ARABA SOUTH
  QA DISI & SOUTHERN DESERT
  SARHAN


## 4. Build Jordan-Domain Grid Mask and Basin ID

In [4]:
sample_nc = MODEL_DIR / f"merged_{MODELS[0]}.nc"
with xr.open_dataset(sample_nc) as ds:
    lats      = ds["lat"].values
    lons      = ds["lon"].values
    all_times = ds["time"].values

n_lat, n_lon = len(lats), len(lons)
print(f"Grid      : {n_lat} lat × {n_lon} lon = {n_lat*n_lon:,} cells")
print(f"Full axis : {len(all_times):,} steps  "
      f"({pd.Timestamp(all_times[0]).date()} → {pd.Timestamp(all_times[-1]).date()})")
print()
print("Building Jordan mask and basin_id grid ...")

jordan_basin_names = gdf_jordan["BASIN_NAME"].tolist()
jordan_mask        = np.zeros((n_lat, n_lon), dtype=bool)
basin_id_grid      = np.full((n_lat, n_lon), -1, dtype=np.int8)

t_start = _time.time()
for li, lat in enumerate(lats):
    for lj, lon in enumerate(lons):
        pt = Point(lon, lat)
        for bi, bname in enumerate(jordan_basin_names):
            if gdf_jordan.iloc[bi].geometry.contains(pt):
                jordan_mask[li, lj]   = True
                basin_id_grid[li, lj] = bi
                break

elapsed = _time.time() - t_start
n_inside  = int(jordan_mask.sum())
n_outside = int((~jordan_mask).sum())
print(f"  Done ({elapsed:.0f}s)")
print(f"  Inside Jordan  : {n_inside:,} cells")
print(f"  Outside Jordan : {n_outside:,} cells")
print()
print("Cells per basin:")
for bi, bname in enumerate(jordan_basin_names):
    n = int((basin_id_grid == bi).sum())
    print(f"  {bname:<32} : {n:>4} cells")

Grid      : 85 lat × 75 lon = 6,375 cells
Full axis : 40,177 steps  (1961-01-01 → 2070-12-31)

Building Jordan mask and basin_id grid ...
  Done (3s)
  Inside Jordan  : 840 cells
  Outside Jordan : 5,535 cells

Cells per basin:
  HAMMAD                           :  176 cells
  YARMOUK (JORDAN)                 :   11 cells
  J.VALLEY-YARMOUK TRIANGLE        :    0 cells
  JORDAN VALLY (JORDAN)            :    5 cells
  N.R.S.W                          :   11 cells
  AZRAQ (JORDAN)                   :  113 cells
  AMMAN ZARQA (JORDAN)             :   33 cells
  S.R.S.W                          :    6 cells
  MUJIB                            :   61 cells
  D.S.R.S.W                        :   15 cells
  W. ARABA NORTH                   :   27 cells
  HASA                             :   22 cells
  JAFER                            :  116 cells
  WADI ARABA SOUTH                 :   55 cells
  QA DISI & SOUTHERN DESERT        :   36 cells
  SARHAN                           :  153 cells


## 5. Basin ID Attribute String

In [5]:
basin_id_labels = "; ".join(
    f"{bi}={bname}" for bi, bname in enumerate(jordan_basin_names)
)
print("basin_id encoding:")
for bi, bname in enumerate(jordan_basin_names):
    n = int((basin_id_grid == bi).sum())
    print(f"  {bi:>2} = {bname:<32}  cells={n:>4}")
print(f"  -1 = outside Jordan domain  (cells={n_outside:,})")

basin_id encoding:
   0 = HAMMAD                            cells= 176
   1 = YARMOUK (JORDAN)                  cells=  11
   2 = J.VALLEY-YARMOUK TRIANGLE         cells=   0
   3 = JORDAN VALLY (JORDAN)             cells=   5
   4 = N.R.S.W                           cells=  11
   5 = AZRAQ (JORDAN)                    cells= 113
   6 = AMMAN ZARQA (JORDAN)              cells=  33
   7 = S.R.S.W                           cells=   6
   8 = MUJIB                             cells=  61
   9 = D.S.R.S.W                         cells=  15
  10 = W. ARABA NORTH                    cells=  27
  11 = HASA                              cells=  22
  12 = JAFER                             cells= 116
  13 = WADI ARABA SOUTH                  cells=  55
  14 = QA DISI & SOUTHERN DESERT         cells=  36
  15 = SARHAN                            cells= 153
  -1 = outside Jordan domain  (cells=5,535)


## 6. Generate the Six Jordan-Wide NetCDF Files

In [6]:
all_times_pd = pd.DatetimeIndex(all_times)

for period_label, y_start, y_end in PERIODS:
    out_filename = f"Jordan_6model_ensemble_ssp585_{period_label}.nc"
    out_path     = OUTPUT_DIR / out_filename

    if out_path.exists():
        print(f"[SKIP] {out_filename} already exists.")
        continue

    time_mask    = (all_times_pd.year >= y_start) & (all_times_pd.year <= y_end)
    time_idxs    = np.where(time_mask)[0]
    period_times = all_times[time_idxs]
    n_t          = len(time_idxs)

    print(f"[PROCESSING] {out_filename}")
    print(f"  Period    : {y_start}–{y_end}  ({n_t:,} time steps)")
    t0 = _time.time()

    # Accumulate all 6 models — equal weight at every Jordan cell
    ens_sum = np.zeros((n_t, n_lat, n_lon), dtype=np.float32)

    for model in MODELS:
        nc_path = MODEL_DIR / f"merged_{model}.nc"
        with xr.open_dataset(nc_path) as ds_m:
            pr_model = ds_m[PR_VAR].isel(time=time_idxs).values
        # Only accumulate inside Jordan
        ens_sum[:, jordan_mask] += pr_model[:, jordan_mask]
        del pr_model
        print(f"  {model:<22} accumulated")

    # Divide by 6 inside Jordan; NaN outside
    ens_pr = np.where(
        (~jordan_mask)[np.newaxis, :, :],
        np.nan,
        ens_sum / float(N_MODELS)
    ).astype(np.float32)
    del ens_sum

    pr_da = xr.DataArray(
        ens_pr,
        dims=["time", "lat", "lon"],
        coords={"time": period_times, "lat": lats, "lon": lons},
        name=PR_VAR,
        attrs={
            "standard_name": "precipitation_flux",
            "long_name"    : "Equal-Weight 6-Model Ensemble Mean Precipitation",
            "units"        : "mm/day",
            "cell_methods" : "time: mean",
            "comment"      : (
                "Arithmetic mean of all 6 RICAAR CMIP6 models under SSP5-8.5. "
                "All models contribute equally to every Jordan grid cell. "
                "Grid cells outside Jordan are NaN."
            ),
        }
    )

    basin_id_da = xr.DataArray(
        basin_id_grid,
        dims=["lat", "lon"],
        coords={"lat": lats, "lon": lons},
        name="basin_id",
        attrs={
            "long_name"       : "Jordan hydrological basin index",
            "basin_id_labels" : basin_id_labels,
            "note"            : "-1 = outside Jordan domain",
        }
    )

    ds_out = xr.Dataset(
        {PR_VAR: pr_da, "basin_id": basin_id_da},
        attrs={
            "title"       : f"Jordan-Wide 6-Model Equal-Weight Ensemble Precipitation {y_start}-{y_end}",
            "institution" : "Jordan University of Science and Technology",
            "scenario"    : "SSP5-8.5",
            "period"      : f"{y_start}-{y_end}",
            "domain"      : "Jordan  (0.1deg x 0.1deg)",
            "original_data": "RICAAR SMHI-HCLIM-ALADIN-38, SMHI-MIdAS021",
            "ensemble_method": "Equal-weight arithmetic mean of all 6 CMIP6 models",
            "Conventions" : "CF-1.7",
            "history"     : (
                f"Created {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
                f"Hussien et al. Jordan CMIP6 precipitation study"
            ),
        }
    )

    ds_out["lat"].attrs  = {"units": "degrees_north", "long_name": "latitude",  "axis": "Y"}
    ds_out["lon"].attrs  = {"units": "degrees_east",  "long_name": "longitude", "axis": "X"}
    ds_out["time"].attrs = {"long_name": "time"}

    encoding = {
        PR_VAR: {
            "dtype"     : "float32",
            "zlib"      : True,
            "complevel" : 4,
            "_FillValue": np.float32(1e+20),
        },
        "basin_id": {
            "dtype"     : "int8",
            "zlib"      : True,
            "complevel" : 4,
            "_FillValue": np.int8(-1),
        },
    }

    ds_out.to_netcdf(out_path, encoding=encoding, format="NETCDF4")
    del ens_pr, ds_out

    elapsed_s = _time.time() - t0
    size_mb   = out_path.stat().st_size / 1024 / 1024
    print(f"  Saved     : {out_filename}  ({size_mb:.1f} MB)  [{elapsed_s:.0f}s]")
    print()

print("=" * 65)
print("ALL SIX FILES COMPLETE")
print("=" * 65)

[PROCESSING] Jordan_6model_ensemble_ssp585_1961_2070.nc
  Period    : 1961–2070  (40,177 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  EC-Earth3-Veg          accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accumulated
  Saved     : Jordan_6model_ensemble_ssp585_1961_2070.nc  (97.8 MB)  [21s]

[PROCESSING] Jordan_6model_ensemble_ssp585_1995_2014.nc
  Period    : 1995–2014  (7,305 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  EC-Earth3-Veg          accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accumulated
  Saved     : Jordan_6model_ensemble_ssp585_1995_2014.nc  (19.5 MB)  [4s]

[PROCESSING] Jordan_6model_ensemble_ssp585_2015_2020.nc
  Period    : 2015–2020  (2,192 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  EC-Earth3-Veg          accum

## 7. Verify Output Files

In [7]:
nc_out_files = sorted(OUTPUT_DIR.glob("Jordan_6model_ensemble_ssp585_*.nc"))
print(f"Files found: {len(nc_out_files)}")
print()
print(f"{'File':<52} {'Period':>18} {'Steps':>7} {'Jordan cells':>13} {'pr mean':>10} {'Size MB':>8}")
print("-" * 115)

for f in nc_out_files:
    ds = xr.open_dataset(f)
    n_t      = len(ds["time"])
    t0_str   = str(pd.Timestamp(ds["time"].values[0]).date())
    t1_str   = str(pd.Timestamp(ds["time"].values[-1]).date())
    pr_vals  = ds[PR_VAR].values
    pr_valid = pr_vals[~np.isnan(pr_vals)]
    bid      = ds["basin_id"].values
    n_jordan = int((bid >= 0).sum())
    size_mb  = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<50} {t0_str} → {t1_str} {n_t:>7,} {n_jordan:>13,} {pr_valid.mean():>10.4f} {size_mb:>8.1f}")
    ds.close()

print()
print("Cross-check vs NB19 (3-model): pr means should be similar for the reference period.")

Files found: 6

File                                                             Period   Steps  Jordan cells    pr mean  Size MB
-------------------------------------------------------------------------------------------------------------------
  Jordan_6model_ensemble_ssp585_1961_2070.nc         1961-01-01 → 2070-12-31  40,177           840     0.3166     97.8
  Jordan_6model_ensemble_ssp585_1995_2014.nc         1995-01-01 → 2014-12-31   7,305           840     0.3032     19.5
  Jordan_6model_ensemble_ssp585_2015_2020.nc         2015-01-01 → 2020-12-31   2,192           840     0.3297      5.9
  Jordan_6model_ensemble_ssp585_2021_2040.nc         2021-01-01 → 2040-12-31   7,305           840     0.3103     19.7
  Jordan_6model_ensemble_ssp585_2041_2060.nc         2041-01-01 → 2060-12-31   7,305           840     0.3141     19.7
  Jordan_6model_ensemble_ssp585_2061_2070.nc         2061-01-01 → 2070-12-31   3,652           840     0.3332      9.9

Cross-check vs NB19 (3-model): pr means